# Qari-OCR on Colab — Arabic OCR benchmark

Runs top to bottom in **one pass**, no restarts. That is the whole design of the
first cell: every version-sensitive package is pinned and installed *before
anything imports it*, because the restarts are only ever needed when a library
is replaced underneath a kernel that already loaded it.

**Before you run anything: Runtime → Change runtime type → T4 GPU → Save.**

Scores identically to `tesseract_benchmark.ipynb` — same normalisation, same
CER/WER, same space-ratio signal — so run both on the same pages of the same PDF
and the two tables read as one.

## 1. Pinned install — run this first, before any other cell

Every version here is deliberate, and each one was learned the hard way:

| pin | why |
| --- | --- |
| `transformers==4.51.3` | Qari predates transformers v5, which reworked how pre-quantised checkpoints load. Colab ships v5. |
| `peft==0.15.2` | Must match the transformers quantisation contract; it is peft that attaches the LoRA. |
| `bitsandbytes==0.45.5` | The base is a pre-quantised 4-bit checkpoint, so bnb is mandatory, not optional. |
| **pillow — untouched** | Colab imports PIL at startup. Upgrading it mid-kernel leaves a mixed install and fails later as `cannot import name '_Ink' from 'PIL._typing'`, from a `transformers` import, pointing nowhere near Pillow. |
| **torch — untouched** | Replacing torch under a live kernel guarantees a restart. |

`--upgrade` with no upper bound is what caused this: a model published against
4.x quietly got a major version it was never tested with.

In [ ]:
# No imports above this line, on purpose.
import sys

_loaded = [m for m in ("transformers", "peft", "bitsandbytes") if m in sys.modules]

!pip -q install "transformers==4.51.3" "peft==0.15.2" "bitsandbytes==0.45.5" \
                qwen-vl-utils pymupdf accelerate

if _loaded:
    raise SystemExit(
        f"{', '.join(_loaded)} was already imported before being replaced.\n"
        "Runtime -> Restart session, then run from this cell. The install has\n"
        "already happened, so it is a no-op afterwards and the notebook then runs\n"
        "in a single pass. In a fresh session this branch never fires."
    )

print("installed. pillow and torch left at Colab's versions on purpose.")
print("The dependency-conflict warnings about diffusers/gradio are expected and")
print("harmless: they want a newer huggingface-hub, and this notebook uses neither.")

## 2. GPU and versions

In [ ]:
import torch, transformers, peft, bitsandbytes

if not torch.cuda.is_available():
    raise SystemExit(
        "No GPU. Runtime -> Change runtime type -> T4 GPU -> Save, then rerun.\n"
        "Qari on CPU is not slow, it is unusable: minutes per page, not seconds."
    )

print("transformers", transformers.__version__, "| peft", peft.__version__,
      "| bitsandbytes", bitsandbytes.__version__)
print("gpu:", torch.cuda.get_device_name(0),
      f"{torch.cuda.get_device_properties(0).total_memory/1024**3:.0f} GB")

# A T4 is Turing and has no bfloat16 -- that needs Ampere (capability 8.0+).
# Asking for bf16 on a T4 is emulated at best, and the timings then mean nothing.
MAJOR, _ = torch.cuda.get_device_capability()
DTYPE = torch.bfloat16 if MAJOR >= 8 else torch.float16
print("dtype:", str(DTYPE).split(".")[-1])

## 3. The PDF

In [ ]:
# --- get a PDF in ---------------------------------------------------------
# Drag a file into the file browser on the left and set PDF_PATH, or run this
# and pick one. For anything large, mount Drive instead -- an upload widget on
# a 25 MB book is slower than Drive and dies on a flaky connection.
from google.colab import files
import os

PDF_PATH = "/content/book.pdf"

if not os.path.exists(PDF_PATH):
    uploaded = files.upload()
    name = next(iter(uploaded))
    os.rename(name, PDF_PATH)

import pymupdf
doc = pymupdf.open(PDF_PATH)
print(f"{doc.page_count} pages")

# Which pages to test. A handful is enough: these engines are seconds per page
# and the numbers stabilise quickly. Pick from the middle -- front matter and
# title pages are not representative of body text.
FIRST, LAST = 40, 47
PAGES = list(range(FIRST, min(LAST + 1, doc.page_count)))
print("testing pages", PAGES)


## 4. Scoring

In [ ]:
# --- scoring -------------------------------------------------------------
# Both notebooks score the same way so their numbers can sit in one table.
# CER and WER are both reported because on fragmented Arabic they disagree:
# splitting `اليسار` into `ا ليسا ر` changes no letters and destroys every
# word, so CER barely moves while WER collapses -- and WER is the one that
# predicts whether retrieval works.
import unicodedata, re

def normalize(text: str) -> str:
    """NFKC, strip bidi controls, collapse whitespace.

    Presentation forms (U+FB50-FDFF, U+FE70-FEFF) fold to typed letters here,
    so an engine is not punished for emitting a form the pipeline normalises
    away before indexing anyway.
    """
    text = unicodedata.normalize("NFKC", text)
    text = text.translate(dict.fromkeys(map(ord, "\u200e\u200f\u202a\u202b\u202c\u202d\u202e\u2066\u2067\u2068\u2069")))
    return re.sub(r"\s+", " ", text).strip()

def levenshtein(a, b):
    if a == b: return 0
    if not a: return len(b)
    if not b: return len(a)
    prev = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        cur = [i]
        for j, cb in enumerate(b, 1):
            cur.append(min(prev[j] + 1, cur[j - 1] + 1, prev[j - 1] + (ca != cb)))
        prev = cur
    return prev[-1]

def cer(truth, hyp):
    t, h = normalize(truth), normalize(hyp)
    return 1.0 if not t else min(1.0, levenshtein(t, h) / len(t))

def wer(truth, hyp):
    t, h = normalize(truth).split(), normalize(hyp).split()
    return 1.0 if not t else min(1.0, levenshtein(t, h) / len(t))

def space_ratio(text):
    """Intrinsic quality signal for when there is no ground truth.

    Healthy Arabic prose sits around 0.13-0.22. Far below means words fused
    together; far above means they were split mid-word, which is the failure
    this whole exercise is about.
    """
    t = normalize(text)
    return 0.0 if not t else t.count(" ") / len(t)


## 5. Load the model

**What is actually being loaded**, because it is not one model:

```
NAMAA-Space/Qari-OCR-0.2.2.1-VL-2B-Instruct      132 MB   a LoRA adapter, nothing else
    └─ base_model_name_or_path:
       unsloth/qwen2-vl-2b-instruct-unsloth-bnb-4bit      a PRE-quantised 4-bit base
```

Qari is a set of low-rank deltas trained with QLoRA against an already-4-bit
base. Both halves have to be reconstructed, in that order, or peft attaches a
LoRA to layers that are not what it expects and you get

```
AttributeError: 'Parameter' object has no attribute 'compress_statistics'
```

raised from `peft/tuners/lora/bnb.py`, many frames below a line that only says
`from_pretrained`. That message means: peft believed the layer was
`bitsandbytes.nn.Linear4bit` and found an ordinary `torch.nn.Parameter` in it.

Passing your own `BitsAndBytesConfig` does **not** help — the base carries its
own, and transformers says so in a `UserWarning` before ignoring yours.

In [ ]:
import json, os, time
from huggingface_hub import snapshot_download
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor
from peft import PeftModel

ADAPTER = "NAMAA-Space/Qari-OCR-0.2.2.1-VL-2B-Instruct"

adapter_path = snapshot_download(ADAPTER)
BASE = json.load(open(os.path.join(adapter_path, "adapter_config.json")))["base_model_name_or_path"]
print("adapter:", ADAPTER)
print("base   :", BASE)

t0 = time.perf_counter()

# No quantization_config passed: the base checkpoint is already 4-bit and its
# own config wins. Ours would be silently ignored.
base_model = Qwen2VLForConditionalGeneration.from_pretrained(BASE, device_map="auto")

model = PeftModel.from_pretrained(base_model, adapter_path)
model.eval()

processor = AutoProcessor.from_pretrained(
    BASE,
    min_pixels=256 * 28 * 28,
    # Qwen2-VL turns a page into 28x28 patches. Too low a cap does not error --
    # it returns about five characters a page, which reads as the model being
    # bad rather than the image being unreadable.
    max_pixels=6400 * 28 * 28,
)

print(f"loaded in {time.perf_counter()-t0:.0f}s")
print(f"VRAM allocated: {torch.cuda.memory_allocated()/1024**3:.1f} GB")

## 5b. Did quantisation actually apply?

Worth thirty seconds, because the failure above is silent until peft trips over
it. Count layer types per tower rather than sampling one layer: **the vision
tower is deliberately left unquantised** — quantising a vision encoder costs
accuracy for little memory — so a plain `Linear` there is correct and says
nothing about the language tower, which is the half peft patches.

Reading it:

- language layers `Linear4bit / Params4bit` → correct, everything above worked
- language layers `Linear4bit / Parameter` → the shells were built but never
  filled; a transformers/bitsandbytes mismatch, and the pins in cell 1 are what
  fix it
- language layers plain `Linear` → bnb never engaged at all

In [ ]:
from collections import Counter

kinds, examples = Counter(), {}
for name, mod in model.named_modules():
    cls = type(mod).__name__
    if cls.startswith("Linear") and hasattr(mod, "weight"):
        tower = "visual" if name.startswith("visual") or ".visual." in name else "language"
        key = (tower, cls, type(mod.weight).__name__)
        kinds[key] += 1
        examples.setdefault(key, name)

for (tower, cls, wcls), n in sorted(kinds.items()):
    print(f"{n:5d}  {tower:8s} {cls:14s} weight={wcls:10s}  e.g. {examples[(tower, cls, wcls)]}")

quantised = any(t == "language" and w == "Params4bit" for (t, _, w) in kinds)
print("\nlanguage tower quantised:", quantised)
if not quantised:
    print("If the run below produces nonsense, this is why — not the model.")

## 6. Read the pages

In [ ]:
import io, re, time
import pymupdf
from PIL import Image

DPI = 300

def render(page_no, dpi=DPI):
    d = pymupdf.open(PDF_PATH)
    try:
        pix = d[page_no].get_pixmap(dpi=dpi)
        return Image.open(io.BytesIO(pix.tobytes("png"))).convert("RGB")
    finally:
        d.close()

def strip_markup(text: str) -> str:
    """Qari emits light HTML by design.

    Scoring the tags as if they were OCR errors is a mistake I made once and it
    cost the model a factor of four: 0.233 WER became 0.063 once the markup it
    is supposed to produce stopped being counted against it.
    """
    text = re.sub(r"<[^>]+>", " ", text)
    return re.sub(r"\s+", " ", text).strip()

PROMPT = "Below is the image of one page of a document. Extract all text exactly as it appears."

@torch.inference_mode()
def read(page_no):
    image = render(page_no)
    messages = [{"role": "user", "content": [
        {"type": "image", "image": image},
        {"type": "text", "text": PROMPT},
    ]}]
    chat = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = processor(text=[chat], images=[image], return_tensors="pt").to(model.device)
    out = model.generate(**inputs, max_new_tokens=2048, do_sample=False)
    trimmed = out[0][len(inputs.input_ids[0]):]
    return processor.decode(trimmed, skip_special_tokens=True)

texts, per_page = {}, {}
for n in PAGES:
    t0 = time.perf_counter()
    texts[n] = strip_markup(read(n))
    per_page[n] = time.perf_counter() - t0
    print(f"  page {n}: {per_page[n]:5.1f}s  {len(texts[n]):5d} chars  space ratio {space_ratio(texts[n]):.3f}")

print(f"\nmean {sum(per_page.values())/len(per_page):.1f} s/page on {torch.cuda.get_device_name(0)}")

## 7. What it read

Check this by eye before trusting any number below it. Diacritics present, words
not split mid-token, inline English intact — those are the things Qari is meant
to be better at. A handful of characters per page means the pixel cap is wrong,
not the model.

In [ ]:
for n in PAGES[:2]:
    print(f"--- page {n} ---")
    print(texts[n][:600])
    print()

## 8. Scores

In [ ]:
TRUTH = {
    # 40: "اليسار حينئذ بديدو ومعناه الهاربة ...",
}

print(f"{'page':>5}  {'chars':>6}  {'s/page':>7}  {'space ratio':>12}")
for n, t in texts.items():
    print(f"{n:>5}  {len(normalize(t)):>6}  {per_page[n]:>7.1f}  {space_ratio(t):>12.3f}")

if TRUTH:
    print(f"\n{'page':>5}  {'CER':>6}  {'WER':>6}")
    for n, truth in TRUTH.items():
        print(f"{n:>5}  {cer(truth, texts[n]):>6.3f}  {wer(truth, texts[n]):>6.3f}")
else:
    print("\nNo ground truth supplied — CER/WER skipped. Paste one page's correct")
    print("text into TRUTH to get them; one careful page beats ten guessed ones.")

## 9. Reading the result against Tesseract

Put these numbers beside `tesseract_benchmark.ipynb` run on the **same pages of
the same PDF**.

| | `tesseract-best` | `qari` |
| --- | ---: | ---: |
| WER, this project's fixtures | 0.172 | **0.063** |
| s/page, real book page | ~2 fast CPU · **~5.3 on the t3.medium deployment box** | ~30 on a T4 |
| needs | nothing | ~5 GB VRAM fp16, ~2 GB at 4-bit |

Roughly an order of magnitude slower per page for about three times the
accuracy. Worth it for a document you care about; not for a 200-page book in
bulk. That is why `tesseract-best` is what ingestion runs and Qari is the
deliberate path.

**Two caveats on the numbers you just produced.**

The base here is 4-bit, while the 0.063 WER in `FINDINGS.md` was measured at
fp16 on a faster card. Quantisation changes throughput and, marginally, output —
so record that this run was 4-bit rather than filing it against that figure.

And seconds-per-page is a property of the engine *and the machine*. Quoting a
laptop's figure at a server is how a 214-page Arabic book came to need 1136s
against a 540s task budget: the same fixed Python loop takes 0.44s on a laptop
and 1.48s on the t3.medium, a 3.4x gap. Read the ratios between machines, not
any single absolute.